# Crossing-Normalized Structural Training (CNST) Dependency Parser

This notebook implements the algorithm from the manuscript: a graph-based dependency parser with contextual token embeddings, bilinear arc scoring, Chu--Liu/Edmonds maximum spanning arborescence decoding, and the crossing-normalized calibration loss.

The implementation is designed to be runnable in two modes:

1. **Synthetic demo mode** (default): trains on a tiny in-notebook treebank so the algorithm can be inspected without downloading data.
2. **UD/BERT mode**: optional hooks show where to plug in Universal Dependencies CoNLL-U files and a Hugging Face BERT encoder.


In [ ]:
import math
import random
from dataclasses import dataclass
from typing import Dict, Iterable, List, Sequence, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
random.seed(7)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE


## Data structures

A dependency tree stores one head for every token. Token index `0` is the artificial ROOT, while surface tokens are indexed from `1..n`. The `heads` list has length `n + 1`; `heads[0]` is ignored and each `heads[j]` gives the gold head of token `j`.


In [ ]:
@dataclass
class DependencyExample:
    tokens: List[str]
    heads: List[int]  # heads[0] is ignored; heads[j] is the head of token j

    @property
    def n(self) -> int:
        return len(self.tokens)

    def arcs(self) -> List[Tuple[int, int]]:
        return [(self.heads[j], j) for j in range(1, self.n + 1)]


def make_demo_treebank() -> List[DependencyExample]:
    # Includes a crossing example similar to the manuscript: (2 -> 5) crosses (4 -> 8).
    return [
        DependencyExample(['I', 'saw', 'the', 'man', 'yesterday', 'who', 'you', 'met'],
                          [-1, 2, 0, 4, 2, 2, 8, 8, 4]),
        DependencyExample(['She', 'enjoys', 'small', 'clear', 'examples'],
                          [-1, 2, 0, 5, 5, 2]),
        DependencyExample(['Parsing', 'can', 'handle', 'crossing', 'arcs'],
                          [-1, 3, 3, 0, 5, 3]),
        DependencyExample(['Robust', 'models', 'need', 'structural', 'margins'],
                          [-1, 2, 3, 0, 5, 3]),
    ]

examples = make_demo_treebank()
examples[0].arcs()


## Crossing complexity

The normalized crossing rate is

$$\rho(T)=\frac{C(T)}{\binom{n-1}{2}},$$

with a zero denominator guard for very short sentences. ROOT arcs are ignored in geometric crossing checks because ROOT is not part of the surface word order.


In [ ]:
def crosses(arc_a: Tuple[int, int], arc_b: Tuple[int, int]) -> bool:
    a, b = arc_a
    c, d = arc_b
    if 0 in (a, b, c, d):
        return False
    i, j = sorted((a, b))
    k, ell = sorted((c, d))
    return (i < k < j < ell) or (k < i < ell < j)


def crossing_count(arcs: Sequence[Tuple[int, int]]) -> int:
    count = 0
    for idx in range(len(arcs)):
        for jdx in range(idx + 1, len(arcs)):
            count += int(crosses(arcs[idx], arcs[jdx]))
    return count


def crossing_rate(example: DependencyExample) -> float:
    max_pairs = math.comb(example.n - 1, 2) if example.n >= 3 else 0
    if max_pairs == 0:
        return 0.0
    return crossing_count(example.arcs()) / max_pairs

[(ex.tokens, crossing_count(ex.arcs()), crossing_rate(ex)) for ex in examples]


## Chu--Liu/Edmonds decoding

The decoder below implements maximum spanning arborescence decoding by exhaustive enumeration for clarity in the tiny demo and includes a production hook where a Chu--Liu/Edmonds implementation can be swapped in. For real UD sentences, replace `decode_mst_exhaustive` with an $O(n^2)$ Chu--Liu/Edmonds implementation such as `networkx.maximum_spanning_arborescence` or a parser-library implementation.


In [ ]:
def is_valid_arborescence(heads: List[int]) -> bool:
    n = len(heads) - 1
    if heads[0] != -1:
        return False
    if any(heads[j] == j for j in range(1, n + 1)):
        return False
    # Exactly one ROOT child is common in dependency parsing. Remove this check if your treebank allows more.
    if sum(1 for j in range(1, n + 1) if heads[j] == 0) != 1:
        return False
    for start in range(1, n + 1):
        seen = set()
        node = start
        while node != 0:
            if node in seen:
                return False
            seen.add(node)
            node = heads[node]
            if node < 0 or node > n:
                return False
    return True


def tree_score(scores: torch.Tensor, heads: List[int]) -> torch.Tensor:
    return sum(scores[heads[j], j] for j in range(1, len(heads)))


def decode_mst_exhaustive(scores: torch.Tensor) -> List[int]:
    """Return heads for the highest-scoring valid arborescence.

    `scores[h, m]` is the score for head h -> modifier m. This enumerator is only
    suitable for small teaching examples; use Chu--Liu/Edmonds for full corpora.
    """
    n = scores.shape[0] - 1
    candidates = []
    for m in range(1, n + 1):
        candidates.append([h for h in range(0, n + 1) if h != m])

    best_heads, best_score = None, None
    def backtrack(m: int, heads: List[int]):
        nonlocal best_heads, best_score
        if m == n + 1:
            if is_valid_arborescence(heads):
                score = tree_score(scores, heads)
                if best_score is None or score.item() > best_score.item():
                    best_score = score
                    best_heads = heads.copy()
            return
        for h in candidates[m - 1]:
            heads[m] = h
            backtrack(m + 1, heads)

    backtrack(1, [-1] + [-1] * n)
    if best_heads is None:
        raise ValueError('No valid arborescence found')
    return best_heads


## CNST model

For a lightweight runnable notebook, `CNSTParser` uses a word embedding layer as the contextual encoder. The same scorer can be connected to BERT by replacing the encoder output with word-level BERT vectors produced from first-subword pooling.


In [ ]:
class CNSTParser(nn.Module):
    def __init__(self, vocab_size: int, dim: int = 64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, dim)
        self.root = nn.Parameter(torch.zeros(dim))
        self.W = nn.Parameter(torch.empty(dim, dim))
        nn.init.xavier_uniform_(self.W)

    def encode(self, token_ids: torch.Tensor) -> torch.Tensor:
        token_vectors = self.embedding(token_ids)
        root_vector = self.root.unsqueeze(0)
        return torch.cat([root_vector, token_vectors], dim=0)

    def arc_scores(self, token_ids: torch.Tensor) -> torch.Tensor:
        h = self.encode(token_ids)
        scores = h @ self.W @ h.T
        scores.fill_diagonal_(-1e9)
        scores[:, 0] = -1e9  # ROOT cannot be a modifier.
        return scores


def build_vocab(examples: Iterable[DependencyExample]) -> Dict[str, int]:
    vocab = {'<unk>': 0}
    for ex in examples:
        for tok in ex.tokens:
            key = tok.lower()
            if key not in vocab:
                vocab[key] = len(vocab)
    return vocab


def tensorize(ex: DependencyExample, vocab: Dict[str, int]) -> torch.Tensor:
    return torch.tensor([vocab.get(tok.lower(), 0) for tok in ex.tokens], dtype=torch.long, device=DEVICE)

vocab = build_vocab(examples)
model = CNSTParser(len(vocab), dim=64).to(DEVICE)


## CNST losses

The total training objective is

$$L_{CNST}=L_{arc}+\gamma\max(0, lpha + etaho(T)-\Delta_	heta(x,T)),$$

where $\Delta_	heta$ is the gold-tree score minus the decoded competitor score.


In [ ]:
def arc_cross_entropy(scores: torch.Tensor, heads: List[int]) -> torch.Tensor:
    losses = []
    for modifier in range(1, len(heads)):
        target = torch.tensor([heads[modifier]], device=scores.device)
        logits = scores[:, modifier].unsqueeze(0)
        losses.append(F.cross_entropy(logits, target))
    return torch.stack(losses).sum()


def cnst_loss(model: CNSTParser, ex: DependencyExample, vocab: Dict[str, int],
              alpha: float = 0.2, beta: float = 2.0, gamma: float = 0.5):
    token_ids = tensorize(ex, vocab)
    scores = model.arc_scores(token_ids)

    predicted_heads = decode_mst_exhaustive(scores.detach())
    gold_score = tree_score(scores, ex.heads)
    predicted_score = tree_score(scores, predicted_heads)
    delta = gold_score - predicted_score

    l_arc = arc_cross_entropy(scores, ex.heads)
    rho = crossing_rate(ex)
    l_cal = torch.relu(torch.tensor(alpha + beta * rho, device=scores.device) - delta)
    return l_arc + gamma * l_cal, {
        'arc_loss': float(l_arc.detach().cpu()),
        'cal_loss': float(l_cal.detach().cpu()),
        'rho': rho,
        'delta': float(delta.detach().cpu()),
        'predicted_heads': predicted_heads,
    }


## Training loop

The loop follows Algorithm 1 in the manuscript: encode tokens, score arcs, decode a tree, compute arc loss, compute $\rho(T)$, compute the margin gap, and update with the combined CNST objective.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)

history = []
for epoch in range(40):
    random.shuffle(examples)
    total = 0.0
    for ex in examples:
        optimizer.zero_grad()
        loss, info = cnst_loss(model, ex, vocab)
        loss.backward()
        optimizer.step()
        total += float(loss.detach().cpu())
    history.append(total / len(examples))
    if epoch % 10 == 0 or epoch == 39:
        print(f'epoch={epoch:02d} mean_loss={history[-1]:.4f}')


## Evaluation by crossing-rate bin

This utility reports unlabeled attachment score (UAS) and exact match (EM), stratified by crossing complexity.


In [ ]:
def evaluate(model: CNSTParser, examples: Sequence[DependencyExample], vocab: Dict[str, int]):
    bins = {'rho=0': [], '0<rho<=0.05': [], 'rho>0.05': []}
    rows = []
    with torch.no_grad():
        for ex in examples:
            scores = model.arc_scores(tensorize(ex, vocab))
            pred = decode_mst_exhaustive(scores)
            correct = sum(int(pred[j] == ex.heads[j]) for j in range(1, ex.n + 1))
            uas = correct / ex.n
            em = int(correct == ex.n)
            rho = crossing_rate(ex)
            key = 'rho=0' if rho == 0 else ('0<rho<=0.05' if rho <= 0.05 else 'rho>0.05')
            bins[key].append((uas, em))
            rows.append((ex.tokens, rho, uas, em, pred, ex.heads))
    summary = {}
    for key, vals in bins.items():
        if vals:
            summary[key] = {
                'sentences': len(vals),
                'UAS': sum(v[0] for v in vals) / len(vals),
                'EM': sum(v[1] for v in vals) / len(vals),
            }
    return summary, rows

summary, rows = evaluate(model, examples, vocab)
summary


In [ ]:
for tokens, rho, uas, em, pred, gold in rows:
    print(' '.join(tokens))
    print(f'  rho={rho:.3f} UAS={uas:.2f} EM={em}')
    print(f'  gold={gold[1:]}')
    print(f'  pred={pred[1:]}')


## Optional: using BERT and Universal Dependencies

To run the same objective on UD data, replace the toy encoder and data loader with the following building blocks:

```python
# pip install transformers conllu networkx
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')
bert = AutoModel.from_pretrained('bert-base-cased')

def bert_word_vectors(words):
    encoded = tokenizer(words, is_split_into_words=True, return_tensors='pt')
    outputs = bert(**encoded).last_hidden_state[0]
    word_ids = encoded.word_ids()
    vectors = []
    for word_index in range(len(words)):
        first_subword = word_ids.index(word_index)
        vectors.append(outputs[first_subword])
    return torch.stack(vectors)
```

For production-scale decoding, replace `decode_mst_exhaustive` with Chu--Liu/Edmonds. The rest of the CNST objective remains unchanged.
